# baseline v2 — SSAFY 16기 1회차 AI 챌린지

RTX 5060 Ti (16GB) / Windows 기준. baseline v3에서 다음을 수정했습니다.

| # | 수정 내용 | 이유 |
|---|---|---|
| 1 | **라벨 마스킹** — 정답 토큰 1개에만 loss | 기존엔 프롬프트·이미지패드 토큰 전체를 학습해 신호가 희석됨 |
| 2 | **logit 비교 추론** — `generate()` 제거 | 파싱 실패 시 `"a"` 폴백이 사라지고, 추론 30~60분 → 5분 |
| 3 | **전체 5,073개 학습** | 기존 200개는 optimizer step이 45회뿐 |
| 4 | **train 층화 홀드아웃 + accuracy 검증** | 기존 valid loss는 모델 선택 신호로 무의미 |
| 5 | **dev pseudo-label 추가** (동의 3/5 이상) | 규정상 dev 사용 허용. 라벨 정확도 83~97% |
| 6 | **bf16 통일 / GradScaler 제거** | GradScaler는 fp16 전용. compute_dtype과 autocast가 엇갈려 있었음 |
| 7 | **선지 순서 TTA** | 위치 편향 완화 |
| 8 | **경로 수정 + 제출 검증** | `/content/`, `content/submission.csv`가 Windows에서 실패 |

## 학습·추론 정합성

학습과 추론이 **정확히 같은 형태**가 되도록 맞췄습니다.

```
학습:  [프롬프트 + assistant 헤더] + "b"    ← 마지막 토큰 1개에만 loss
추론:  [프롬프트 + assistant 헤더]         ← 마지막 위치 logits에서 a/b/c/d만 비교
```

정답 뒤에 `<|im_end|>`를 붙이지 않으므로 **단일 토큰 4지 분류** 문제가 되고, 추론과 완전히 일치합니다.

# 0. 환경 확인

베이스라인의 `pip install` 셀은 이미 실행하셨다고 가정합니다.
아직이라면 원본 노트북의 Cell 4·6을 먼저 돌리세요.

⚠️ **5060 Ti는 Blackwell(sm_120)** 이라 torch nightly cu128이 필요합니다.
`pip install ... --upgrade`가 nightly를 안정판으로 덮어썼는지 아래에서 확인합니다.

In [ ]:
import torch, sys
print("python :", sys.version.split()[0])
print("torch  :", torch.__version__)          # +cu128 / dev 문자열이 있어야 함
print("cuda   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device :", torch.cuda.get_device_name())
    cap = torch.cuda.get_device_capability()
    print("compute capability :", cap)        # 5060 Ti = (12, 0)
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM   : {total/1e9:.1f} GB (free {free/1e9:.1f} GB)")
    assert torch.zeros(8, device="cuda").sum().item() == 0, "CUDA 커널 실행 실패"
    print("\nCUDA 커널 정상 동작 ✅")

# 1. 설정

## 실측으로 확인된 것 (5060 Ti 16GB)

| | 홀드아웃/LB |
|---|---|
| 텍스트 전용 (이미지 미사용) | 0.5655 |
| 베이스라인 LB — 384px + `generate()` | 0.68979 |
| **v2 zero-shot — 512px + logit 비교** | **0.7438** |
| 파인튜닝 기여분 | **0.00%p** |

**개선은 전부 해상도와 추론 방식에서 나왔고 학습에서는 나오지 않았습니다.**
그래서 기본값은 "학습을 아낀다"는 방향으로 잡혀 있습니다.

### 속도 실측
| 설정 | s/batch |
|---|---|
| 512px, batch 4, grad ckpt ON | **3.75** |
| 위와 같으나 VRAM이 넘쳐 시스템 RAM으로 스필 | **7.65** |

512px에서 4,464개 1 epoch = 약 70분 → 예산 초과라 `MAX_TRAIN_SAMPLES=2400`으로 제한했습니다.

> ⚠️ **`USE_GRAD_CKPT`를 끄지 마세요.** 메모리를 더 쓰게 되고, VRAM이 차면 Windows WDDM이
> 조용히 시스템 RAM으로 스필해 2배 이상 느려집니다. 여유가 있어 보여도 학습 중에 찹니다.

## 권장 실행 순서

1. **`SKIP_TRAIN = True`로 먼저 제출물 확보** — 학습 없이 zero-shot으로 추론·제출 (약 20분)
2. **해상도 스윕 셀** 실행 — 384/512/640을 홀드아웃 300건으로 비교 (약 5분, 모델 재로드 불필요)
3. 더 좋은 해상도가 나오면 그걸로 전체 추론 후 재제출
4. 시간이 남으면 `SKIP_TRAIN = False`로 학습

`MAX_TRAIN_MIN`이 학습 시간 상한입니다. 초과하면 중단하고 best 체크포인트로 추론까지
진행하므로 **제출물 없이 끝나는 일은 없습니다.**

In [ ]:
from pathlib import Path

# ========== 스모크 테스트 ==========
# True  : 소량(학습 96 / 검증 32 / 테스트 64)으로 전 과정 5분 내 검증
# False : 본 실행
SMOKE_TEST      = False                  # 스모크 통과 확인됨 -> 본 실행
MAX_TRAIN_MIN   = 50                     # 학습 시간 상한(분). 넘으면 중단하고 추론·제출 진행

ROOT = Path("../data").resolve()         # train.csv · train/ · test/ 가 있는 폴더 (저장소의 data/)

# ---------- 모델 ----------
# 3B bf16 640px zero-shot = 홀드아웃 0.7504 / LB 0.76823 (측정 완료)
# 7B는 bf16이면 16.6GB라 16GB 카드에 안 들어갑니다 -> 4bit(약 5.5GB) 필수
MODEL_ID  = "Qwen/Qwen2.5-VL-7B-Instruct"
USE_4BIT  = True
IMAGE_PX  = 640                          # 스윕 결과 384<512<640, 수확 체감 구간

# ---------- LoRA ----------
LORA_R          = 16                     # 베이스라인 8 -> 16
LORA_ALPHA      = 32
LORA_DROPOUT    = 0.05
LORA_TARGETS    = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]

# ---------- 학습 ----------
SKIP_TRAIN      = True                   # True면 학습을 건너뛰고 zero-shot 상태로 추론·제출
RUN_SWEEP       = False                  # 해상도 스윕(셀 25) 실행 여부. 7B에서는 너무 느립니다
EPOCHS          = 1                      # 5060 Ti 실측상 1 epoch이 현실적 상한
MAX_TRAIN_SAMPLES = 2400                 # 학습에 쓸 최대 샘플 수 (시간 예산 맞춤, None이면 전체)
USE_GRAD_CKPT   = True                   # ★ 끄지 마세요. VRAM이 넘치면 WDDM이 시스템 RAM으로
                                         #   스필해 2배 이상 느려집니다 (실측 3.75 -> 7.65 s/batch)
BATCH_SIZE      = 4
GRAD_ACCUM      = 2                      # 유효 배치 = 8
LR              = 1e-4
WARMUP_RATIO    = 0.03
MAX_GRAD_NORM   = 1.0
NUM_WORKERS     = 0                      # Windows 노트북은 0이 안전
EVAL_EVERY      = 150                    # optimizer step 기준 (1 epoch = 558 step)
VAL_RATIO       = 0.12                   # train에서 떼는 홀드아웃 비율

# ---------- dev pseudo-label ----------
# 실행 #1은 False 권장: 학습 데이터가 4,464 -> 7,972로 늘어 시간이 1.8배가 되고,
# dev는 카운팅 비율 71%(test는 34%)라 오히려 해로울 수 있어 효과가 불확실합니다.
# 먼저 핵심 수정만으로 LB를 확인한 뒤, 실행 #2에서 True로 A/B 하세요.
USE_DEV         = False
DEV_MIN_AGREE   = 3                      # 3 -> 3,508개(라벨정확도 ~83%) / 4 -> 902개(~97%)
DEV_WEIGHT      = 0.5                    # dev 샘플의 loss 가중치

# ---------- 증강 ----------
USE_AUG         = True

# ---------- 추론 ----------
INFER_BATCH     = 4                      # 8은 512px에서 VRAM 여유가 100MB대까지 떨어짐
N_TTA           = 1                      # 1=끄기, 2~4=선지 순서 순환 치환 앙상블
QUICK_N         = 300                    # 설정 스윕용 빠른 평가 표본 수

SEED = 42

# ---------- 실행별 고유 경로 ----------
# 설정을 파일명에 넣어, 실행마다 어댑터/제출물이 서로 덮어쓰지 않게 합니다.
#  - 스모크 테스트가 저장한 어댑터를 본 실행이 잘못 불러오는 사고 방지
#  - 좋은 점수가 나온 제출 파일을 다음 실행이 지우는 사고 방지
MODEL_SHORT = MODEL_ID.split("/")[-1].replace("Qwen2.5-VL-", "").replace("-Instruct", "")

def run_tag():
    """현재 설정으로 태그 생성. IMAGE_PX가 바뀌면(스윕 등) 다시 불러 반영하세요."""
    return (f"{'smoke' if SMOKE_TEST else 'full'}_{MODEL_SHORT}"
            f"_{'4bit' if USE_4BIT else 'bf16'}_{IMAGE_PX}px"
            f"_{'zs' if SKIP_TRAIN else 'ft'}"
            f"_{'dev' if USE_DEV else 'nodev'}")

RUN_TAG  = run_tag()
SAVE_DIR = ROOT / f"lora_{RUN_TAG}"
print("RUN_TAG   :", RUN_TAG)
print("어댑터    :", SAVE_DIR.name)
print("※ 제출 파일명은 셀 34에서 그 시점의 IMAGE_PX로 다시 계산합니다")

# 2. 라이브러리 · 시드

In [ ]:
import os, math, random, json, time, gc
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
from typing import Any, List
from PIL import Image, ImageEnhance
from tqdm.auto import tqdm

from transformers import AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

Image.MAX_IMAGE_PIXELS = None
device = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)

print("device:", device)

# 3. 데이터

### 검증 설계
- **`dev.csv`는 검증셋으로 쓰지 않습니다.** 교육생 응답이라 상호 일치도가 37.5%(랜덤 25%)에 불과하고,
  카운팅 문항 비율도 71%로 test(33.8%)와 완전히 다릅니다.
- 대신 **train을 층화 분할**합니다. train과 test는 카운팅 비율이 34.4% vs 33.8%로 거의 동일합니다.
- 층화 기준 = `정답 위치(a~d) × 문항 유형(카운팅/기타)`

### dev pseudo-label (기본 OFF)
5인 응답의 다수결을 라벨로 씁니다. 동의 수준별 행 수는 2/5 903 · 3/5 2,606 · 4/5 902 입니다.

> ⚠️ 여기서 "동의 n/5면 정답일 확률 몇 %" 같은 추정치는 **쓰지 않습니다.**
> "5명이 같은 문항에 독립적으로 답했다"고 가정하면 만장일치 비율이 최소 0.0039여야 하는데
> 실제는 1/4413 = 0.00023 으로 **최소 17배 차이**가 납니다. 그 가정이 성립하지 않으므로
> 거기서 유도한 사후확률도 신뢰할 수 없습니다.
> dev를 학습에 넣을지는 **홀드아웃 정확도로 직접 A/B** 해서 결정하세요.

In [ ]:
train_df = pd.read_csv(ROOT / "train.csv")
test_df  = pd.read_csv(ROOT / "test.csv")
sample_sub = pd.read_csv(ROOT / "sample_submission.csv")

CNT_PAT = r"몇 개|몇개|개입니까|개인가요|몇 가지"
def is_counting(s):
    return s.str.contains(CNT_PAT, regex=True)

# ---------- 층화 홀드아웃 ----------
from sklearn.model_selection import train_test_split
strat = train_df["answer"].astype(str) + "_" + is_counting(train_df["question"]).astype(str)
tr_idx, va_idx = train_test_split(
    np.arange(len(train_df)), test_size=VAL_RATIO, random_state=SEED, stratify=strat
)
tr_part = train_df.iloc[tr_idx].reset_index(drop=True)
va_part = train_df.iloc[va_idx].reset_index(drop=True)
tr_part["weight"] = 1.0

print(f"train {len(tr_part)} / valid {len(va_part)}")
print("  valid 카운팅 비율 %.1f%% (test 33.8%%)" % (100*is_counting(va_part.question).mean()))

# ---------- dev pseudo-label ----------
if USE_DEV:
    dev_df = pd.read_csv(ROOT / "dev.csv")
    acols = [f"answer{i}" for i in range(1, 6)]
    maj, agree = [], []
    for _, r in dev_df.iterrows():
        vals = [r[c] for c in acols if pd.notna(r[c]) and str(r[c]) in "abcd"]
        if not vals:
            maj.append(None); agree.append(0); continue
        top, n = pd.Series(vals).value_counts().idxmax(), pd.Series(vals).value_counts().max()
        maj.append(top); agree.append(int(n))
    dev_df["answer"] = maj
    dev_df["_agree"] = agree
    keep = dev_df[(dev_df["_agree"] >= DEV_MIN_AGREE) & dev_df["answer"].notna()].copy()
    keep["weight"] = DEV_WEIGHT
    keep = keep[["id", "path", "question", "a", "b", "c", "d", "answer", "weight"]]
    print(f"\ndev pseudo-label {len(keep)}개 추가 (동의 >= {DEV_MIN_AGREE}/5, weight {DEV_WEIGHT})")
    print("  dev 카운팅 비율 %.1f%%  <- test와 분포가 다르니 weight로 완화" % (100*is_counting(keep.question).mean()))
    fit_df = pd.concat([tr_part, keep], ignore_index=True)
else:
    fit_df = tr_part

fit_df = fit_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

# ---------- 시간 예산에 맞춰 학습 샘플 수 제한 ----------
# 512px 기준 실측 3.75 s/batch(batch 4) -> 1.07 샘플/초.
# 2,400개면 약 37분으로 1 epoch을 완주합니다. 전체(4,464)는 70분이라 예산 초과입니다.
if MAX_TRAIN_SAMPLES and len(fit_df) > MAX_TRAIN_SAMPLES:
    fit_df = fit_df.head(MAX_TRAIN_SAMPLES).reset_index(drop=True)
    print(f"학습 샘플을 {MAX_TRAIN_SAMPLES}개로 제한 (시간 예산)")

# ---------- 스모크 테스트면 전부 축소 ----------
if SMOKE_TEST:
    fit_df  = fit_df.head(96).reset_index(drop=True)
    va_part = va_part.head(32).reset_index(drop=True)
    test_df = test_df.head(64).reset_index(drop=True)
    EPOCHS, EVAL_EVERY = 1, 10
    print("\n" + "="*52)
    print("  SMOKE_TEST — 학습 96 / 검증 32 / 테스트 64")
    print("  통과하면 SMOKE_TEST=False 로 바꾸고 처음부터 재실행")
    print("  (이 모드의 제출 파일은 64행뿐이라 캐글에 올리면 안 됩니다)")
    print("="*52)

print(f"\n최종 학습 데이터: {len(fit_df)}개 / 검증 {len(va_part)}개 / 테스트 {len(test_df)}개")

# 4. 프롬프트

베이스라인과 거의 동일하되, **정답 뒤에 `<|im_end|>`를 붙이지 않는 것**이 핵심입니다.
마지막 토큰이 항상 정답 글자가 되어 마스킹과 추론 위치가 정확히 맞아떨어집니다.

In [ ]:
SYSTEM_INSTRUCT = (
    "You are a helpful visual question answering assistant. "
    "Answer using exactly one letter among a, b, c, or d. No explanation."
)

def build_mc_prompt(question, a, b, c, d):
    return (
        f"{question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."
    )

def build_messages(row, img):
    # 베이스라인과 동일하게 실제 PIL 이미지를 넣는다.
    # (chat_template은 플레이스홀더만 찍지만, 검증된 형태를 그대로 유지)
    o = [str(row[k]) for k in "abcd"]
    return [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
        {"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": build_mc_prompt(str(row["question"]), *o)},
        ]},
    ]

# 5. 모델 · LoRA

베이스라인 대비 수정점:
- `bnb_4bit_compute_dtype`을 **bfloat16**으로 (autocast와 통일)
- `prepare_model_for_kbit_training`에 **`use_reentrant=False`** 전달.
  베이스라인은 이 함수가 켠 gradient checkpointing을 다시 `gradient_checkpointing_enable()`로
  덮어써서 reentrant 모드로 되돌렸고, 이 경우 LoRA gradient가 흐르지 않을 수 있습니다.
- `device_map="auto"` 사용 시 **`.to(device)`를 호출하지 않습니다.**

In [ ]:
# ---------- 같은 커널에서 재실행할 때의 이중 로드 방지 ----------
# 이 셀을 두 번 실행하면 이전 모델이 VRAM에 남은 채 두 번째 모델이 올라가,
# 공간이 부족해지면 accelerate가 일부 레이어를 조용히 오프로드(meta 디바이스)합니다.
# 그러면 학습 중 "expected device meta but got cuda:0" 로 backward가 실패합니다.
for _v in ["model", "base_model"]:
    if _v in globals():
        del globals()[_v]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    _free = torch.cuda.mem_get_info()[0] / 1e9
    print(f"로드 전 VRAM 여유: {_free:.1f} GB")
    assert _free > 8.0, (
        f"VRAM 여유가 {_free:.1f} GB뿐입니다. 커널을 재시작(Restart Kernel)한 뒤 "
        "처음부터 다시 실행하세요. 이대로 진행하면 오프로드가 일어나 학습이 실패합니다."
    )

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=IMAGE_PX * IMAGE_PX,
    max_pixels=IMAGE_PX * IMAGE_PX,
    trust_remote_code=True,
)
tok = processor.tokenizer

# device_map="auto" 는 VRAM이 모자라면 말없이 CPU/디스크로 오프로드합니다.
# {"": 0} 으로 고정하면 전부 GPU에 올리거나, 안 되면 OOM으로 분명하게 실패합니다.
load_kw = dict(device_map={"": 0}, trust_remote_code=True, torch_dtype=torch.bfloat16)
if USE_4BIT:
    load_kw["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,   # baseline: float16 (autocast와 불일치)
    )

base_model = AutoModelForVision2Seq.from_pretrained(MODEL_ID, **load_kw)

# 오프로드가 일어났는지 즉시 확인 — meta/cpu 에 남은 파라미터가 있으면 학습이 불가능합니다
_bad = [(n, str(p.device)) for n, p in base_model.named_parameters()
        if p.device.type in ("meta", "cpu")]
assert not _bad, (
    f"GPU 밖에 남은 파라미터 {len(_bad)}개 (예: {_bad[:2]}). "
    "VRAM 부족으로 오프로드된 상태입니다. 커널을 재시작하고 다시 실행하세요."
)
print(f"모든 파라미터 GPU 적재 확인 ✅")

if USE_4BIT:
    base_model = prepare_model_for_kbit_training(
        base_model,
        use_gradient_checkpointing=USE_GRAD_CKPT,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    )
elif USE_GRAD_CKPT:
    base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    base_model.enable_input_require_grads()

if not USE_GRAD_CKPT:
    base_model.enable_input_require_grads()
    print("gradient checkpointing OFF — 속도 우선 (VRAM 여유가 있을 때만)")

base_model.config.use_cache = False

model = get_peft_model(base_model, LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none",
    target_modules=LORA_TARGETS, task_type="CAUSAL_LM",
))

# ★ 학습 파라미터는 반드시 fp32
# bf16은 eps가 약 0.0078이라, lr 1e-4 수준의 미세한 AdamW 업데이트가 그냥 사라집니다.
# (4bit 경로는 prepare_model_for_kbit_training이 이미 처리하지만 안전하게 한 번 더)
n_cast = 0
for _, p in model.named_parameters():
    if p.requires_grad and p.dtype != torch.float32:
        p.data = p.data.to(torch.float32); n_cast += 1
print(f"fp32로 변환한 학습 파라미터 텐서: {n_cast}개")

model.print_trainable_parameters()

n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert n_train > 0, "학습 가능한 파라미터가 0입니다 — LoRA 적용 실패"
assert all(p.dtype == torch.float32 for p in model.parameters() if p.requires_grad), \
    "학습 파라미터에 fp32가 아닌 것이 남아 있습니다"

### 선지 토큰 확인

`a`, `b`, `c`, `d`가 각각 **단일 토큰**이어야 logit 비교가 성립합니다. 반드시 통과해야 합니다.

In [ ]:
CHOICES = list("abcd")
CHOICE_IDS = []
for ch in CHOICES:
    ids = tok.encode(ch, add_special_tokens=False)
    assert len(ids) == 1, f"'{ch}' 가 단일 토큰이 아님: {ids}"
    CHOICE_IDS.append(ids[0])
assert len(set(CHOICE_IDS)) == 4, "선지 토큰이 중복됨"
CHOICE_IDS = torch.tensor(CHOICE_IDS)
print("선지 토큰 id:", dict(zip(CHOICES, CHOICE_IDS.tolist())), "✅")

if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    used = (total - free) / 1e9
    print(f"\n모델 로드 후 VRAM: {used:.1f} / {total/1e9:.1f} GB 사용  (여유 {free/1e9:.1f} GB)")
    if free / 1e9 < 4:
        print("⚠️ 여유가 4GB 미만입니다. 학습 중 OOM 위험 —"
              " BATCH_SIZE를 2로, GRAD_ACCUM을 4로 낮추세요.")

# 6. Dataset · Collator

## 라벨 마스킹 (핵심 수정)

베이스라인:
```python
enc["labels"] = enc["input_ids"].clone()      # 전부 학습 -> 정답 신호 희석
```

v2 — **마지막 실제 토큰(=정답 글자) 1개만 남기고 전부 `-100`**:
```python
labels = torch.full_like(input_ids, -100)
last = attention_mask.sum(1) - 1              # right padding 기준
labels[arange(B), last] = input_ids[arange(B), last]
```

## 증강 정책

카운팅 문항이 34%라 **객체 수를 바꾸는 증강은 금지**입니다.

| 기법 | 사용 | 이유 |
|---|---|---|
| 밝기·대비·채도 | ✅ | 정답 불변 |
| 소각도 회전(±8°) | ✅ | 정답 불변 |
| **crop** | ❌ | 객체가 잘려나가 카운팅 정답이 깨짐 |
| **좌우 반전** | ❌ | "왼쪽/오른쪽" 위치 문항(2.4%)의 정답이 뒤집힘 |

In [ ]:
def augment(img):
    if random.random() < 0.7:
        img = ImageEnhance.Brightness(img).enhance(random.uniform(0.75, 1.25))
    if random.random() < 0.7:
        img = ImageEnhance.Contrast(img).enhance(random.uniform(0.80, 1.20))
    if random.random() < 0.5:
        img = ImageEnhance.Color(img).enhance(random.uniform(0.80, 1.20))
    if random.random() < 0.3:
        img = img.rotate(random.uniform(-8, 8), resample=Image.BILINEAR, fillcolor=(255, 255, 255))
    return img


class VQADataset(Dataset):
    """train=True면 정답 글자를 붙이고, False면 프롬프트까지만."""
    def __init__(self, df, train=True, aug=False):
        self.df = df.reset_index(drop=True); self.train = train; self.aug = aug

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(ROOT / row["path"]).convert("RGB")
        if self.aug:
            img = augment(img)
        item = {
            "messages": build_messages(row, img),
            "image": img,
            "weight": float(row["weight"]) if "weight" in row else 1.0,
        }
        if self.train:
            item["gold"] = str(row["answer"]).strip().lower()
        return item


@dataclass
class Collator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts, images, weights = [], [], []
        for s in batch:
            # add_generation_prompt=True -> "...<|im_start|>assistant\n" 까지 생성
            t = self.processor.apply_chat_template(
                s["messages"], tokenize=False, add_generation_prompt=True
            )
            if self.train:
                t = t + s["gold"]          # <|im_end|> 를 붙이지 않는다 (마지막 토큰 = 정답)
            texts.append(t)
            images.append(s["image"])
            weights.append(s["weight"])

        # 학습은 right padding (마지막 실제 토큰 = 정답)
        # 추론은 left  padding (index -1 이 항상 마지막 실제 토큰)
        self.processor.tokenizer.padding_side = "right" if self.train else "left"
        enc = self.processor(text=texts, images=images, padding=True, return_tensors="pt")

        if self.train:
            ids, am = enc["input_ids"], enc["attention_mask"]
            last = am.sum(1) - 1
            labels = torch.full_like(ids, -100)
            r = torch.arange(ids.size(0))
            labels[r, last] = ids[r, last]
            enc["labels"] = labels
            enc["_last"] = last
        enc["_weight"] = torch.tensor(weights, dtype=torch.float32)
        return enc

### 마스킹 검증 — 반드시 정답 글자 1개만 남아야 합니다

In [ ]:
_dbg = DataLoader(VQADataset(fit_df.head(4), train=True), batch_size=4,
                  collate_fn=Collator(processor, True))
_b = next(iter(_dbg))
print("input_ids :", tuple(_b["input_ids"].shape))
for i in range(4):
    lab = _b["labels"][i]
    kept = lab[lab != -100]
    print(f"  [{i}] 학습 대상 토큰 {len(kept)}개 -> {tok.decode(kept)!r}"
          f"   (정답 {fit_df.iloc[i]['answer']!r})")
    assert len(kept) == 1, "정답 토큰이 1개가 아닙니다"
    assert tok.decode(kept).strip() == str(fit_df.iloc[i]["answer"]).strip()
print("\n마스킹 정상 ✅  — 베이스라인은 여기서 수천 개가 나왔습니다")
del _dbg, _b

# 7. 검증 — logit 기반 정확도

`generate()`를 쓰지 않습니다. **마지막 위치의 logits에서 a/b/c/d 4개만 비교**합니다.

- 생성 루프가 없어 훨씬 빠름
- 파싱 실패가 원천적으로 불가능 (베이스라인의 `"a"` 폴백 제거)
- 4개 확률을 그대로 얻으므로 TTA 앙상블이 자연스러움

In [ ]:
@torch.no_grad()
def predict_probs(df, batch_size=None, desc="predict"):
    """각 행에 대해 a/b/c/d 확률 (N,4) 반환."""
    bs = batch_size or INFER_BATCH
    ds = VQADataset(df, train=False, aug=False)
    dl = DataLoader(ds, batch_size=bs, shuffle=False,
                    collate_fn=Collator(processor, False), num_workers=NUM_WORKERS)
    model.eval()
    out = []
    cid = CHOICE_IDS.to(device)
    for batch in tqdm(dl, desc=desc, leave=False):
        w = batch.pop("_weight", None)
        batch = {k: v.to(device) for k, v in batch.items() if isinstance(v, torch.Tensor)}
        logits = model(**batch).logits[:, -1, :]        # left padding -> -1 이 마지막 실제 토큰
        out.append(F.softmax(logits[:, cid].float(), dim=-1).cpu())
    return torch.cat(out).numpy()


def evaluate(df, desc="valid"):
    p = predict_probs(df, desc=desc)
    pred = np.array(CHOICES)[p.argmax(1)]
    gold = df["answer"].astype(str).str.strip().values
    acc = (pred == gold).mean()
    m = is_counting(df["question"]).values
    return {
        "acc": acc,
        "acc_count": (pred[m] == gold[m]).mean() if m.sum() else float("nan"),
        "acc_other": (pred[~m] == gold[~m]).mean() if (~m).sum() else float("nan"),
        "pred": pred,
    }

### 학습 전 zero-shot 성능

파인튜닝을 시작하기 전에 사전학습 모델이 그냥 몇 점인지 봅니다.
**이 값이 텍스트 전용 베이스라인(0.5655)보다 낮으면** 프롬프트나 이미지 처리에 문제가 있는 것입니다.

In [ ]:
t0 = time.time()
zs = evaluate(va_part, desc="zero-shot")
print(f"zero-shot 정확도 : {zs['acc']:.4f}   ({time.time()-t0:.0f}초)")
print(f"  카운팅 : {zs['acc_count']:.4f}")
print(f"  기타   : {zs['acc_other']:.4f}")
print(f"\n비교 기준 — 랜덤 0.2500 / 텍스트전용 0.5655")

## ⚡ 해상도 스윕 (선택 — 매우 저렴합니다)

**`IMAGE_PX`는 프로세서 설정일 뿐이라, 바꿔도 모델을 다시 로드할 필요가 없습니다.**
홀드아웃 일부만 평가하면 해상도별 성능을 몇 분 만에 비교할 수 있습니다.

지금까지 확인된 바로는 **개선의 대부분이 학습이 아니라 해상도에서** 나왔습니다
(384px 베이스라인 LB 0.690 → 512px zero-shot 홀드아웃 0.7438).
학습에 70분을 쓰기 전에 이걸 먼저 돌려보는 게 기대값이 높습니다.

⚠️ 해상도를 올리면 VRAM도 늘어납니다. 768px에서 OOM이면 `INFER_BATCH`를 2로 낮추세요.

In [ ]:
if not RUN_SWEEP:
    print("RUN_SWEEP=False — 해상도 스윕을 건너뜁니다.")
    print("3B 측정 완료: 384px 0.7307 / 512px 0.7455 / 640px 0.7504  (609건, ±2.0%p)")
else:
    # 원래 설정을 보관
    _orig_px, _orig_proc = IMAGE_PX, processor
    _quick = va_part.head(QUICK_N).reset_index(drop=True)
    print(f"홀드아웃 {len(_quick)}건으로 해상도별 zero-shot 비교\n")

    sweep = {}
    for _px in [384, 512, 640]:
        processor = AutoProcessor.from_pretrained(
            MODEL_ID, min_pixels=_px*_px, max_pixels=_px*_px, trust_remote_code=True)
        _t = time.time()
        _r = evaluate(_quick, desc=f"{_px}px")
        sweep[_px] = _r["acc"]
        print(f"  {_px}px : acc {_r['acc']:.4f}  "
              f"(카운팅 {_r['acc_count']:.3f} / 기타 {_r['acc_other']:.3f})  {time.time()-_t:.0f}초")
        gc.collect(); torch.cuda.empty_cache()

    _best_px = max(sweep, key=sweep.get)
    print(f"\n최고: {_best_px}px ({sweep[_best_px]:.4f})")
    print(f"※ 표본 {len(_quick)}건이라 표준오차 약 ±{100*(0.25/len(_quick))**0.5:.1f}%p — "
          "차이가 그보다 작으면 우연입니다.")

    # 최고 해상도로 프로세서 고정 (원복하려면 processor = _orig_proc)
    IMAGE_PX  = _best_px
    processor = AutoProcessor.from_pretrained(
        MODEL_ID, min_pixels=IMAGE_PX*IMAGE_PX, max_pixels=IMAGE_PX*IMAGE_PX, trust_remote_code=True)
    tok = processor.tokenizer
    print(f"\nprocessor를 {IMAGE_PX}px로 고정했습니다.")

# 8. 학습

`SKIP_TRAIN = True` 로 두면 이 셀은 아무것도 하지 않고 넘어갑니다.
zero-shot 성능만으로 제출하고 싶을 때 쓰세요.

베이스라인 대비 수정점:
- **GradScaler 제거** (fp16 전용이라 bf16에선 의미 없음)
- **샘플별 가중치** 지원 → dev pseudo-label에 낮은 weight 적용
- **gradient clipping** 추가
- **마지막 자투리 배치의 gradient도 반영** (베이스라인은 버려짐)
- **주기적 accuracy 검증 + 최고 성능 어댑터 저장**

In [ ]:
def run_training():
    """학습 수행. (best_acc, global_step, hist) 반환."""
    gc.collect(); torch.cuda.empty_cache()
    print(f"학습 시작 전 VRAM 여유: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

    train_loader = DataLoader(
        VQADataset(fit_df, train=True, aug=USE_AUG),
        batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=Collator(processor, True), num_workers=NUM_WORKERS, drop_last=False,
    )

    steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
    total_steps = steps_per_epoch * EPOCHS
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=0.01)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, int(total_steps * WARMUP_RATIO), total_steps
    )
    print(f"총 optimizer step: {total_steps}  /  총 batch: {len(train_loader)*EPOCHS}")
    print(f"MAX_TRAIN_MIN={MAX_TRAIN_MIN}분 안에 완주하려면 "
          f"{MAX_TRAIN_MIN*60/(len(train_loader)*EPOCHS):.2f} s/batch 이하가 필요합니다.")
    print("→ 진행바의 s/batch가 이보다 크면 상한에 걸려 중간에 끊깁니다. "
          "그래도 best 체크포인트로 추론은 진행됩니다.")


    def masked_weighted_loss(logits, labels, last, weight):
        """정답 위치 1개에 대한 CE를 샘플 가중치로 가중평균."""
        B = logits.size(0)
        r = torch.arange(B, device=logits.device)
        # 위치 t의 logits가 위치 t+1을 예측하므로 한 칸 당김
        sel = logits[r, last - 1, :].float()
        tgt = labels[r, last]
        ce = F.cross_entropy(sel, tgt, reduction="none")
        return (ce * weight).sum() / weight.sum().clamp(min=1e-8)


    set_seed(SEED)
    best_acc, global_step, hist = zs["acc"], 0, []

    # 동일 설정으로 재실행할 때 이전 어댑터가 남아 있으면, 이번 학습이 zero-shot을
    # 못 넘겼을 때 셀 29가 옛 어댑터를 불러옵니다. 항상 "이번 실행의 결과"만 남기도록 제거.
    SAVE_DIR.mkdir(parents=True, exist_ok=True)
    _stale = SAVE_DIR / "adapter_model.safetensors"
    if _stale.exists():
        _stale.unlink(); print(f"이전 실행의 어댑터 제거: {_stale.name}")

    t_start, stop = time.time(), False          # 시간 상한 감시

    for epoch in range(EPOCHS):
        if stop: break
        model.train()
        running, seen = 0.0, 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", unit="batch")
        optimizer.zero_grad(set_to_none=True)

        for step, batch in enumerate(pbar, start=1):
            w    = batch.pop("_weight").to(device)
            last = batch.pop("_last").to(device)
            batch = {k: v.to(device) for k, v in batch.items() if isinstance(v, torch.Tensor)}
            labels = batch.pop("labels")

            out = model(**batch)
            loss = masked_weighted_loss(out.logits, labels, last, w)
            (loss / GRAD_ACCUM).backward()
            running += loss.item(); seen += 1

            is_last = (step == len(train_loader))
            if step % GRAD_ACCUM == 0 or is_last:          # 자투리 배치도 반영
                torch.nn.utils.clip_grad_norm_(params, MAX_GRAD_NORM)
                optimizer.step(); scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1
                pbar.set_postfix({"loss": f"{running/max(seen,1):.3f}",
                                  "lr": f"{scheduler.get_last_lr()[0]:.2e}"})
                running, seen = 0.0, 0

                if global_step % EVAL_EVERY == 0:
                    r = evaluate(va_part, desc=f"valid@{global_step}")
                    hist.append((global_step, r["acc"]))
                    tqdm.write(f"  [step {global_step}] valid acc {r['acc']:.4f} "
                               f"(카운팅 {r['acc_count']:.3f} / 기타 {r['acc_other']:.3f})")
                    if r["acc"] > best_acc:
                        best_acc = r["acc"]
                        model.save_pretrained(SAVE_DIR); processor.save_pretrained(SAVE_DIR)
                        tqdm.write(f"    -> best 갱신, 저장 ({SAVE_DIR.name})")
                    model.train()

                # ---- 시간 상한 ----
                if (time.time() - t_start) / 60 > MAX_TRAIN_MIN:
                    tqdm.write(f"\n⏰ {MAX_TRAIN_MIN}분 경과 — 학습을 중단하고 추론 단계로 넘어갑니다.")
                    stop = True
                    break

        r = evaluate(va_part, desc=f"valid ep{epoch+1}")
        hist.append((global_step, r["acc"]))
        print(f"[Epoch {epoch+1}] valid acc {r['acc']:.4f} "
              f"(카운팅 {r['acc_count']:.3f} / 기타 {r['acc_other']:.3f})")
        if r["acc"] > best_acc:
            best_acc = r["acc"]
            model.save_pretrained(SAVE_DIR); processor.save_pretrained(SAVE_DIR)
            print(f"  -> best 갱신, 저장")
        model.train()

    print(f"\n최고 valid 정확도: {best_acc:.4f}   (zero-shot {zs['acc']:.4f})")
    return best_acc, global_step, hist


if SKIP_TRAIN:
    best_acc, global_step, hist = zs["acc"], 0, []
    print("SKIP_TRAIN=True — 학습을 건너뜁니다. zero-shot 상태로 셀 30부터 진행하세요.")
else:
    best_acc, global_step, hist = run_training()

# 9. 학습 결과 확인

**valid 정확도가 zero-shot보다 낮으면 제출하지 마세요.** 학습이 모델을 망친 것이므로
LR을 낮추거나(5e-5) epoch을 줄여야 합니다.

In [ ]:
if hist:
    print("step   valid acc")
    for s, a in hist:
        print(f"{s:5d}   {a:.4f}")
LB_BASELINE = 0.68979      # baseline v3(200개 학습, 384px)의 실제 리더보드 점수
print(f"\nzero-shot     : {zs['acc']:.4f}")
print(f"best (학습후) : {best_acc:.4f}   ({(best_acc-zs['acc'])*100:+.2f}%p)")
print(f"\n기준선 — 텍스트전용 0.5655 / 베이스라인 LB {LB_BASELINE}")
if best_acc < 0.5655:
    print("⚠️ 텍스트 전용보다 낮습니다 — 이미지 신호가 전혀 활용되지 않고 있습니다.")
elif best_acc < LB_BASELINE:
    print(f"⚠️ 베이스라인 LB({LB_BASELINE})보다 낮습니다 — 제출해도 순위가 떨어집니다.")
    print("   holdout과 LB 사이 오차가 있을 수 있으나, 큰 차이라면 제출을 보류하세요.")
else:
    print(f"✅ 베이스라인 LB를 상회 ({(best_acc-LB_BASELINE)*100:+.2f}%p) — 제출 권장")

# 10. 추론 + 선지 순서 TTA

`N_TTA > 1`이면 선지 순서를 순환 치환해 여러 번 추론하고 확률을 평균냅니다.

```
치환 [1,2,3,0] -> 화면의 (a)에 원래 b, (b)에 원래 c, ... 를 표시
모델이 화면 위치 j를 고르면 -> 원래 선지 perm[j] 에 확률 누적
```

이 데이터는 선지 텍스트 편향이 강해(가장 긴 선지 고르기만으로 42%) 모델이 위치 편향을 학습하기
쉬운데, TTA가 이를 상쇄합니다.

In [ ]:
PERMS = [(0,1,2,3), (1,2,3,0), (2,3,0,1), (3,0,1,2)][:max(1, N_TTA)]

@torch.no_grad()
def predict_with_tta(df, desc="test"):
    total = np.zeros((len(df), 4), dtype=np.float64)
    for pi, perm in enumerate(PERMS):
        d = df.copy()
        if perm != (0, 1, 2, 3):
            orig = df[["a","b","c","d"]].astype(str).values
            for j, src in enumerate(perm):
                d[CHOICES[j]] = orig[:, src]
        p = predict_probs(d, desc=f"{desc} tta{pi+1}/{len(PERMS)}")
        for j, src in enumerate(perm):
            total[:, src] += p[:, j]        # 화면 위치 j -> 원래 선지 src
    return total / len(PERMS)


# 최고 성능 어댑터 가중치를 현재 모델에 덮어쓰기
# (PeftModel.from_pretrained로 다시 감싸면 LoRA가 이중 주입되므로 state_dict만 로드)
from peft import set_peft_model_state_dict
from safetensors.torch import load_file

_ad = SAVE_DIR / "adapter_model.safetensors"
if _ad.exists():
    set_peft_model_state_dict(model, load_file(str(_ad)))
    print("best 어댑터 로드:", _ad.name)
else:
    print("저장된 어댑터 없음 — 현재 메모리 상태 그대로 사용")
model.eval()

# TTA 효과를 valid에서 먼저 확인
if len(PERMS) > 1:
    pv = predict_with_tta(va_part, desc="valid")
    acc_tta = (np.array(CHOICES)[pv.argmax(1)] == va_part["answer"].values).mean()
    print(f"valid — TTA 미적용 {best_acc:.4f} / TTA {len(PERMS)}회 {acc_tta:.4f}")

In [ ]:
t0 = time.time()
probs = predict_with_tta(test_df, desc="test")
preds = np.array(CHOICES)[probs.argmax(1)]
print(f"추론 완료 {time.time()-t0:.0f}초  (베이스라인 30~60분)")
print("예측 분포:", dict(pd.Series(preds).value_counts().sort_index()))
print("평균 최대확률 %.3f  (낮으면 모델이 확신 없음)" % probs.max(1).mean())

# 11. 제출 파일

⚠️ **id는 `test_0001.jpg`처럼 확장자를 포함합니다.** Overview 표의 `test_0001`은 축약 표기이며,
확장자를 떼면 5,074개가 전부 오채점됩니다. `sample_submission.csv` 기준으로 merge하고 검증합니다.

In [ ]:
def write_submission(ids, answers, out_path):
    sub = pd.DataFrame({"id": ids, "answer": answers})
    sub = sample_sub[["id"]].merge(sub, on="id", how="left")
    assert len(sub) == len(sample_sub), f"행 수 불일치 {len(sub)} != {len(sample_sub)}"
    assert (sub["id"].values == sample_sub["id"].values).all(), "id 순서 불일치"
    assert sub["answer"].notna().all(), f"결측 {sub['answer'].isna().sum()}건"
    assert sub["answer"].isin(CHOICES).all(), f"허용 외 값 {set(sub['answer'])-set(CHOICES)}"
    sub.to_csv(out_path, index=False, encoding="utf-8")
    return sub

if SMOKE_TEST:
    print("SMOKE_TEST 모드 — 제출 파일을 만들지 않습니다.")
    print(f"  파이프라인은 여기까지 정상 통과했습니다 (테스트 {len(preds)}건 예측 완료).")
    print("  이제 셀 5에서 SMOKE_TEST = False 로 바꾸고 처음부터 다시 실행하세요.")
else:
    # 스윕 등으로 IMAGE_PX가 바뀌었을 수 있으므로 지금 값으로 파일명을 다시 만듭니다
    SUB_PATH = ROOT / f"submission_{run_tag()}{'_tta'+str(len(PERMS)) if len(PERMS)>1 else ''}.csv"
    sub = write_submission(test_df["id"].values, preds, SUB_PATH)
    print("✅ 저장:", SUB_PATH)
    print(sub.head(3).to_string(index=False))
    print(f"\n검증 통과 — {len(sub)}행 / id 순서 일치 / 값 전부 a~d")
    print(f"참고 valid 정확도 {best_acc:.4f}")

# 12. 오답 분석 (선택)

다음 실험 방향을 정하는 데 씁니다. 카운팅 정확도가 유독 낮으면 → **해상도를 올리세요**(`IMAGE_PX`).
전체가 고르게 낮으면 → 학습량(epoch/LoRA r) 또는 모델 크기 문제입니다.

In [ ]:
r = evaluate(va_part, desc="분석")
va = va_part.copy(); va["pred"] = r["pred"]; va["ok"] = va["pred"] == va["answer"]

print(f"전체 {r['acc']:.4f}\n")
print("문항 유형별:")
print(f"  카운팅 {r['acc_count']:.4f}  ({is_counting(va.question).sum()}건)")
print(f"  기타   {r['acc_other']:.4f}\n")

print("정답 위치별 (편향 확인 — 특정 글자만 높으면 위치 편향):")
print(va.groupby("answer")["ok"].agg(["mean", "size"]).round(3).to_string())

print("\n예측 분포 vs 실제 분포:")
print(pd.DataFrame({"예측": va.pred.value_counts(normalize=True).round(3),
                    "실제": va.answer.value_counts(normalize=True).round(3)}).to_string())

print("\n틀린 문항 예시 5개:")
for _, x in va[~va.ok].head(5).iterrows():
    print(f"  Q: {x['question']}")
    print(f"     a.{x['a']} b.{x['b']} c.{x['c']} d.{x['d']}")
    print(f"     정답 {x['answer']} / 예측 {x['pred']}\n")

In [ ]:
# =====================================================================
#  실행 중인 v2 노트북 "맨 아래에 새 셀로 붙여넣어" 사용하세요.
#  노트북을 다시 로드할 필요 없습니다. (셀 32까지 끝난 뒤 실행)
#
#  아이디어: VLM(0.7504)과 텍스트 전용 모델(0.5714)은 서로 다른 신호를 씁니다.
#           텍스트 모델은 VLM이 무시하는 데이터셋 편향(카운팅 정답이 작은 수에
#           쏠림 등)을 잡으므로, 확률을 블렌딩하면 둘 다보다 높아질 수 있습니다.
#  비용: 홀드아웃 확률 1회 계산(약 4분) 외에는 GPU 사용 없음.
# =====================================================================

# ---------------------------------------------------------------- 셀 A
# VLM 확률을 확보하고 디스크에 저장 (커널이 죽어도 잃지 않도록)
import numpy as np

CH = np.array(CHOICES)
gold_va = va_part["answer"].astype(str).str.strip().values

p_va_vlm = predict_probs(va_part, desc="holdout probs")      # 약 4분
np.save(ROOT / f"vlm_probs_holdout_{run_tag()}.npy", p_va_vlm)

# probs 는 셀 32에서 만들어진 테스트 확률 (5074, 4)
np.save(ROOT / f"vlm_probs_test_{run_tag()}.npy", probs)

acc_vlm = (CH[p_va_vlm.argmax(1)] == gold_va).mean()
print(f"VLM 홀드아웃 정확도 {acc_vlm:.4f}   (저장 완료)")


# ---------------------------------------------------------------- 셀 B
# 텍스트 확률 로드 + α 탐색
p_va_txt = np.load(ROOT / "text_probs_holdout.npy")
p_te_txt = np.load(ROOT / "text_probs_test.npy")
assert len(p_va_txt) == len(va_part) == len(p_va_vlm), "홀드아웃 길이 불일치"
assert len(p_te_txt) == len(probs) == len(test_df),    "테스트 길이 불일치"

def blend(pv, pt, a):
    """로그 공간 가중 기하평균. a=1이면 VLM 단독."""
    q = np.clip(pv, 1e-9, 1) ** a * np.clip(pt, 1e-9, 1) ** (1 - a)
    return q / q.sum(1, keepdims=True)

is_cnt = is_counting(va_part["question"]).values
print(f"{'α':>6} {'전체':>8} {'카운팅':>8} {'기타':>8}")
rows = []
for a in np.arange(0.60, 1.001, 0.025):
    pred = CH[blend(p_va_vlm, p_va_txt, a).argmax(1)]
    ov = (pred == gold_va).mean()
    ct = (pred[is_cnt] == gold_va[is_cnt]).mean()
    ot = (pred[~is_cnt] == gold_va[~is_cnt]).mean()
    rows.append((a, ov, ct, ot))
    print(f"{a:6.3f} {ov:8.4f} {ct:8.4f} {ot:8.4f}")

best_a, best_ov, _, _ = max(rows, key=lambda r: r[1])
gain = best_ov - acc_vlm
print(f"\n최적 α={best_a:.3f}  홀드아웃 {best_ov:.4f}   VLM 단독 {acc_vlm:.4f}   ({gain*100:+.2f}%p)")
print("⚠️ α를 홀드아웃에서 고른 만큼 낙관 편향이 있습니다(약 +0.5~1%p).")
print("   상승폭이 1.5%p 미만이면 블렌딩하지 말고 VLM 단독으로 제출하세요.")


# ---------------------------------------------------------------- 셀 C (선택)
# 문항 유형별로 α를 따로: 텍스트 모델은 카운팅에서 상대적으로 더 쓸모 있습니다
#   (텍스트 카운팅 0.5190 vs 기타 0.5990 / VLM 카운팅 0.6190 vs 기타 0.8195)
best2, sc2 = None, -1
for a_c in np.arange(0.60, 1.001, 0.025):
    for a_o in np.arange(0.60, 1.001, 0.025):
        pred = np.empty(len(va_part), dtype="<U1")
        pred[is_cnt]  = CH[blend(p_va_vlm[is_cnt],  p_va_txt[is_cnt],  a_c).argmax(1)]
        pred[~is_cnt] = CH[blend(p_va_vlm[~is_cnt], p_va_txt[~is_cnt], a_o).argmax(1)]
        s = (pred == gold_va).mean()
        if s > sc2: best2, sc2 = (a_c, a_o), s
print(f"유형별 α — 카운팅 {best2[0]:.3f} / 기타 {best2[1]:.3f}  홀드아웃 {sc2:.4f}")
print("⚠️ 2차원 탐색이라 과적합 위험이 더 큽니다. 단일 α보다 +1%p 이상일 때만 쓰세요.")


# ---------------------------------------------------------------- 셀 C2 (선택)
# 신뢰도 적응형 블렌딩:
#   VLM이 확신하는 문항은 VLM을 그대로 믿고,
#   헷갈려하는 문항(최대확률이 낮음)에서만 텍스트 모델 쪽으로 기울입니다.
#   α(x) = a_hi 일 때 VLM 확신, a_lo 일 때 불확실
conf = p_va_vlm.max(1)
print(f"VLM 확신도 분포: 하위25% {np.percentile(conf,25):.3f} / 중앙 {np.percentile(conf,50):.3f}"
      f" / 상위25% {np.percentile(conf,75):.3f}")
print(f"확신도 상위 절반 정확도 {(CH[p_va_vlm[conf>=np.median(conf)].argmax(1)]==gold_va[conf>=np.median(conf)]).mean():.4f}")
print(f"확신도 하위 절반 정확도 {(CH[p_va_vlm[conf<np.median(conf)].argmax(1)]==gold_va[conf<np.median(conf)]).mean():.4f}")

best3, sc3 = None, -1
for a_lo in np.arange(0.55, 1.001, 0.05):
    for a_hi in np.arange(a_lo, 1.001, 0.05):
        w = (conf - conf.min()) / (conf.max() - conf.min() + 1e-9)
        a = a_lo + (a_hi - a_lo) * w                     # 확신할수록 VLM 비중↑
        q = np.clip(p_va_vlm,1e-9,1)**a[:,None] * np.clip(p_va_txt,1e-9,1)**(1-a[:,None])
        s = (CH[(q/q.sum(1,keepdims=True)).argmax(1)] == gold_va).mean()
        if s > sc3: best3, sc3 = (a_lo, a_hi), s
print(f"신뢰도 적응형 — α_low {best3[0]:.2f} → α_high {best3[1]:.2f}   홀드아웃 {sc3:.4f}")
print("⚠️ 2차원 탐색이라 단일 α보다 +1%p 이상 나을 때만 채택하세요.")


# ---------------------------------------------------------------- 셀 D
# 채택하기로 했다면 제출 파일 생성 (α는 위 결과를 보고 직접 지정하세요)
ALPHA = best_a          # 또는 직접 숫자 입력
p_te_blend = blend(probs, p_te_txt, ALPHA)
preds_blend = CH[p_te_blend.argmax(1)]
print("예측 분포:", dict(pd.Series(preds_blend).value_counts().sort_index()))
print("VLM 단독과 다른 예측:", (preds_blend != preds).sum(), "건")

SUB_BLEND = ROOT / f"submission_{run_tag()}_blend{ALPHA:.3f}.csv"
write_submission(test_df["id"].values, preds_blend, SUB_BLEND)
print("✅ 저장:", SUB_BLEND.name)


In [ ]:
# =====================================================================
#  카운팅 숫자 스코어링 — 검증용 (노트북 맨 아래에 붙여넣기)
#  GPU 약 4분. p_va(홀드아웃 확률)를 재사용하므로 글자 방식은 다시 계산하지 않습니다.
# =====================================================================
import re, numpy as np, pandas as pd, torch
from PIL import Image
from tqdm.auto import tqdm

if "p_va" not in dir():
    p_va = np.load(ROOT / "vlm_probs_holdout_full_7B_4bit_640px_zs_nodev.npy").astype(np.float64)

CH = np.array(CHOICES)

# --- 숫자 토큰 ---
DIGIT_IDS = {}
for d in range(10):
    ids = tok.encode(str(d), add_special_tokens=False)
    assert len(ids) == 1, f"'{d}' 단일 토큰 아님: {ids}"
    DIGIT_IDS[d] = ids[0]
print("숫자 토큰:", DIGIT_IDS)

def opt_num(v):
    s = str(v).strip()
    if s in ("없음", "없다", "0개", "0"): return 0
    if "이상" in s or "미만" in s or s.count("개") > 1: return None
    m = re.fullmatch(r"\s*(\d+)\s*개?\s*", s)
    return int(m.group(1)) if m else None

def numeric_row(row):
    ns = [opt_num(row[k]) for k in "abcd"]
    return (all(n is not None and 0 <= n <= 9 for n in ns) and len(set(ns)) == 4), ns

NUM_SYS = ("You are a visual question answering assistant. "
           "Count carefully and answer with a single digit only. No explanation.")

@torch.no_grad()
def predict_counting(df):
    model.eval()
    out = np.full((len(df), 4), np.nan)
    for s in tqdm(range(0, len(df), INFER_BATCH), desc="숫자 스코어링"):
        sub = df.iloc[s:s + INFER_BATCH]
        texts, images, nss, gis = [], [], [], []
        for j, (_, row) in enumerate(sub.iterrows()):
            ok, ns = numeric_row(row)
            if not ok: continue
            img = Image.open(ROOT / row["path"]).convert("RGB")
            msgs = [{"role": "system", "content": [{"type": "text", "text": NUM_SYS}]},
                    {"role": "user", "content": [{"type": "image", "image": img},
                     {"type": "text", "text": f"{row['question']}\n숫자 한 글자만 출력하세요."}]}]
            texts.append(processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))
            images.append(img); nss.append(ns); gis.append(s + j)
        if not texts: continue
        processor.tokenizer.padding_side = "left"
        enc = processor(text=texts, images=images, padding=True, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items() if isinstance(v, torch.Tensor)}
        lg = model(**enc).logits[:, -1, :].float()
        for i, (gi, ns) in enumerate(zip(gis, nss)):
            out[gi] = torch.softmax(lg[i, [DIGIT_IDS[n] for n in ns]], -1).cpu().numpy()
    return out


# ---- 카운팅 부분집합 준비 (글자 확률은 p_va 재사용) ----
mc = is_counting(va_part["question"]).values
va_cnt = va_part[mc].reset_index(drop=True)
gold_c = va_cnt["answer"].astype(str).str.strip().values
p_let_c = p_va[mc]                                   # 이미 계산해둔 글자 방식 확률
p_txt_c = np.load(ROOT / "text_probs_holdout.npy")[mc]
print(f"홀드아웃 카운팅 {len(va_cnt)}건")

p_num_c = predict_counting(va_cnt)                   # ← GPU 약 4분
ok = ~np.isnan(p_num_c).any(1)
print(f"숫자 스코어링 적용 가능 {ok.sum()}건 ({ok.mean():.1%})")

def norm(p): return p / p.sum(1, keepdims=True)
def bl(pv, pt, a): return norm(np.clip(pv,1e-9,1)**a * np.clip(pt,1e-9,1)**(1-a))
def A(p): return (CH[p.argmax(1)] == gold_c[ok]).mean()

L, T, N = p_let_c[ok], p_txt_c[ok], p_num_c[ok]
print(f"\n적용 가능 {ok.sum()}건에서의 비교  (표준오차 ±{100*(0.25/ok.sum())**0.5:.1f}%p)")
print(f"  글자만                 {A(L):.4f}")
print(f"  글자+텍스트 α=0.55 ★현행 {A(bl(L,T,0.55)):.4f}")
print(f"  숫자만                 {A(N):.4f}")
for a in [0.5, 0.6, 0.7, 0.8]:
    print(f"  숫자+텍스트 α={a:.1f}      {A(bl(N,T,a)):.4f}")
for w in [0.3, 0.5, 0.7]:
    print(f"  글자·숫자 평균 w={w:.1f}    {A(norm(w*N+(1-w)*L)):.4f}")
for w in [0.5]:
    for a in [0.6, 0.7]:
        print(f"  (글자·숫자 w={w}) +텍스트 α={a:.1f}  {A(bl(norm(w*N+(1-w)*L), T, a)):.4f}")
print("\n★ 현행(글자+텍스트)을 +2%p 이상 넘지 못하면 적용하지 마세요.")


In [ ]:
# =====================================================================
#  저신뢰 문항 2지선다 재질의 (binary refinement)
#
#  근거 (홀드아웃 609건):
#    신뢰도 하위 20%  ->  top1 정확도 0.525 / top2 포함률 0.836
#    즉 "정답을 2개로 좁히는 것"은 되는데 "둘 중 고르기"에서 틀립니다.
#    -> 오답 2개를 지우고 2지선다로 다시 물으면 개선 여지가 있습니다.
#
#  셀 1(검증) GPU 약 4분 / 셀 2(적용) GPU 약 35분
# =====================================================================
import numpy as np, pandas as pd, torch
from PIL import Image
from tqdm.auto import tqdm

CH = np.array(CHOICES)
if "p_va" not in dir():
    p_va = np.load(ROOT / "vlm_probs_holdout_full_7B_4bit_640px_zs_nodev.npy").astype(np.float64)
if "probs" not in dir():
    probs = np.load(ROOT / "vlm_probs_test_full_7B_4bit_640px_zs_nodev.npy").astype(np.float64)

def norm(p): return p / p.sum(1, keepdims=True)
def bl(pv, pt, a): return norm(np.clip(pv,1e-9,1)**a * np.clip(pt,1e-9,1)**(1-a))

t_va = np.load(ROOT / "text_probs_holdout.npy")
t_te = np.load(ROOT / "text_probs_test.npy")
P_va = bl(p_va, t_va, 0.75)          # 현재 제출과 동일한 블렌딩
P_te = bl(probs, t_te, 0.75)

BIN_SYS = ("You are a visual question answering assistant. "
           "Answer using exactly one letter, a or b. No explanation.")

@torch.no_grad()
def refine(df, P, idx):
    """idx 행들에 대해 top-2만 남긴 2지선다로 재질의. 갱신된 확률 반환."""
    model.eval()
    out = P.copy()
    ab = [tok.encode(x, add_special_tokens=False)[0] for x in ("a", "b")]
    for s in tqdm(range(0, len(idx), INFER_BATCH), desc="2지선다 재질의"):
        chunk = idx[s:s + INFER_BATCH]
        texts, images, pairs_ = [], [], []
        for i in chunk:
            row = df.iloc[i]
            top2 = sorted(np.argsort(-P[i])[:2])      # 원래 순서 유지(위치 편향 방지)
            o1, o2 = str(row[CHOICES[top2[0]]]), str(row[CHOICES[top2[1]]])
            img = Image.open(ROOT / row["path"]).convert("RGB")
            txt = (f"{row['question']}\n(a) {o1}\n(b) {o2}\n\n"
                   "정답을 반드시 a 또는 b 중 하나의 소문자 한 글자로만 출력하세요.")
            msgs = [{"role": "system", "content": [{"type": "text", "text": BIN_SYS}]},
                    {"role": "user", "content": [{"type": "image", "image": img},
                                                 {"type": "text", "text": txt}]}]
            texts.append(processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))
            images.append(img); pairs_.append((i, top2))
        processor.tokenizer.padding_side = "left"
        enc = processor(text=texts, images=images, padding=True, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items() if isinstance(v, torch.Tensor)}
        lg = model(**enc).logits[:, -1, :].float()
        for k, (i, top2) in enumerate(pairs_):
            q2 = torch.softmax(lg[k, ab], -1).cpu().numpy()
            mass = P[i, top2].sum()                    # 원래 top-2 질량 유지, 내부 순위만 교체
            out[i, top2[0]] = q2[0] * mass
            out[i, top2[1]] = q2[1] * mass
    return norm(out)


# ---------------------------------------------------------------- 셀 1  검증 (약 4분)
gold = va_part["answer"].astype(str).str.strip().values
conf = P_va.max(1)
K = 20                                                  # 하위 K%
thr = np.percentile(conf, K)
idx = np.where(conf <= thr)[0]
print(f"홀드아웃 하위 {K}% = {len(idx)}건 (현재 정확도 {(CH[P_va[idx].argmax(1)]==gold[idx]).mean():.4f})")

P_ref = refine(va_part, P_va, idx)
a_old = (CH[P_va.argmax(1)] == gold).mean()
a_new = (CH[P_ref.argmax(1)] == gold).mean()
s_old = (CH[P_va[idx].argmax(1)] == gold[idx]).mean()
s_new = (CH[P_ref[idx].argmax(1)] == gold[idx]).mean()
print(f"\n대상 {len(idx)}건 : {s_old:.4f} -> {s_new:.4f}  ({(s_new-s_old)*100:+.1f}%p)")
print(f"전체 609건 : {a_old:.4f} -> {a_new:.4f}  ({(a_new-a_old)*100:+.2f}%p)")
print(f"표준오차 대상 ±{100*(0.25/len(idx))**0.5:.1f}%p / 전체 ±2.0%p")
print(f"참고: 이 구간의 top-2 포함률 0.836 = 이론상 상한")
print("\n★ 전체 +1.5%p 이상이면 셀 2 진행")


In [ ]:
for w in [0.1, 0.2, 0.3, 0.5]:
    M = norm((1-w)*P_va + w*P_ref)
    print(f"w={w}: {(CH[M.argmax(1)]==gold).mean():.4f}   (기준 0.8424)")

In [ ]:
# =====================================================================
#  선지 순서 TTA — 저신뢰 구간 검증 (GPU 약 4분)
#  2지선다 실패와의 차이:
#    - 선지 4개를 그대로 유지하고 '순서'만 바꿈 (문제 형식 불변)
#    - TTA는 VLM 확률에만 적용하고, 텍스트 블렌딩은 그 뒤에 다시 얹음
#      (지난번엔 블렌딩된 확률을 덮어써서 텍스트 기여를 날렸음)
# =====================================================================
import numpy as np, pandas as pd, torch
from PIL import Image
from tqdm.auto import tqdm

CH = np.array(CHOICES)
if "p_va" not in dir():
    p_va = np.load(ROOT / "vlm_probs_holdout_full_7B_4bit_640px_zs_nodev.npy").astype(np.float64)
if "probs" not in dir():
    probs = np.load(ROOT / "vlm_probs_test_full_7B_4bit_640px_zs_nodev.npy").astype(np.float64)
t_va = np.load(ROOT / "text_probs_holdout.npy")
t_te = np.load(ROOT / "text_probs_test.npy")

def norm(p): return p / p.sum(1, keepdims=True)
def bl(pv, pt, a=0.75): return norm(np.clip(pv,1e-9,1)**a * np.clip(pt,1e-9,1)**(1-a))

PERM = (1, 2, 3, 0)      # 화면 위치 j 에 원래 선지 PERM[j] 를 표시

@torch.no_grad()
def tta_pass(df, idx, perm=PERM):
    """idx 행을 선지 순서 바꿔 1회 재추론. 원래 선지 기준 (len(idx),4) 확률."""
    model.eval()
    out = np.zeros((len(idx), 4))
    cid = CHOICE_IDS.to(device)
    for s in tqdm(range(0, len(idx), INFER_BATCH), desc="TTA"):
        chunk = idx[s:s + INFER_BATCH]
        texts, images = [], []
        for i in chunk:
            row = df.iloc[i]
            o = [str(row[CHOICES[perm[j]]]) for j in range(4)]
            img = Image.open(ROOT / row["path"]).convert("RGB")
            msgs = [{"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
                    {"role": "user", "content": [{"type": "image", "image": img},
                     {"type": "text", "text": build_mc_prompt(str(row["question"]), *o)}]}]
            texts.append(processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))
            images.append(img)
        processor.tokenizer.padding_side = "left"
        enc = processor(text=texts, images=images, padding=True, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items() if isinstance(v, torch.Tensor)}
        lg = model(**enc).logits[:, -1, :]
        q = torch.softmax(lg[:, cid].float(), -1).cpu().numpy()
        for k in range(len(chunk)):
            for j in range(4):
                out[s + k, perm[j]] = q[k, j]          # 화면 j -> 원래 perm[j]
    return out


# ---- 검증: 홀드아웃 하위 20% ----
gold = va_part["answer"].astype(str).str.strip().values
P_va = bl(p_va, t_va)
conf = P_va.max(1)
idx = np.where(conf <= np.percentile(conf, 20))[0]
print(f"홀드아웃 하위 20% = {len(idx)}건, 현재 {(CH[P_va[idx].argmax(1)]==gold[idx]).mean():.4f}")

q_tta = tta_pass(va_part, idx)                          # VLM 확률만 TTA

p_new = p_va.copy()
p_new[idx] = norm(0.5 * p_va[idx] + 0.5 * q_tta)        # 원본 + 치환 평균
P_new = bl(p_new, t_va)                                 # 텍스트 블렌딩 다시 적용

s_old = (CH[P_va[idx].argmax(1)] == gold[idx]).mean()
s_new = (CH[P_new[idx].argmax(1)] == gold[idx]).mean()
a_old = (CH[P_va.argmax(1)] == gold).mean()
a_new = (CH[P_new.argmax(1)] == gold).mean()
print(f"\n대상 {len(idx)}건 : {s_old:.4f} -> {s_new:.4f}  ({(s_new-s_old)*100:+.1f}%p)")
print(f"전체 609건 : {a_old:.4f} -> {a_new:.4f}  ({(a_new-a_old)*100:+.2f}%p)")
print(f"표준오차 대상 ±{100*(0.25/len(idx))**0.5:.1f}%p / 전체 ±2.0%p")
print("\n★ 전체 +1.5%p 이상이면 tta_2_apply.py 진행 (약 35분)")

In [ ]:
# =====================================================================
#  TTA 적용 — test 저신뢰 구간 (tta_1_check.py 를 같은 커널에서 실행한 뒤)
#  K=40 -> 약 2,029건 / 약 24분
# =====================================================================
K = 40

P_te = bl(probs, t_te)
conf_t = P_te.max(1)
idx_t = np.where(conf_t <= np.percentile(conf_t, K))[0]
print(f"test 하위 {K}% = {len(idx_t)}건 -> 예상 {len(idx_t)*0.71/60:.0f}분")

q_te = tta_pass(test_df, idx_t)                      # VLM 확률만 TTA

p_te_new = probs.copy()
p_te_new[idx_t] = norm(0.5 * probs[idx_t] + 0.5 * q_te)
P_te_new = bl(p_te_new, t_te)                        # 텍스트 블렌딩 다시 적용
preds = CH[P_te_new.argmax(1)]
np.save(ROOT / "vlm7b_probs_test_tta.npy", p_te_new)

sub = pd.DataFrame({"id": test_df["id"].values, "answer": preds})
dup = pd.read_csv(ROOT / "dup_answers.csv").set_index("id")["answer"]
hit = sub["id"].isin(dup.index)
sub.loc[hit, "answer"] = sub.loc[hit, "id"].map(dup)

prev = pd.read_csv(ROOT / "submission_7B_blend_a0.75.csv")
print(f"직전 제출(LB 0.8443) 대비 변경 {(sub.answer.values != prev.answer.values).sum()}건")
print("예측 분포:", dict(sub.answer.value_counts().sort_index()))

write_submission(sub["id"].values, sub["answer"].values, ROOT / f"submission_7B_tta{K}.csv")
print(f"✅ 저장: submission_7B_tta{K}.csv")

In [ ]:
# =====================================================================
#  7B QLoRA 파인튜닝 검증 (자체 완결형 — 노트북 맨 아래 새 셀에 붙여넣기)
#
#  목적: "파인튜닝이 zero-shot(0.8276)을 넘는가?" 하나만 답한다.
#  설계: 이미 로드된 7B / 640px 모델을 그대로 쓰므로 재로드 불필요.
#        검증은 홀드아웃 앞 400건으로 하고, 같은 400건의 zero-shot 점수는
#        저장된 .npy 에서 공짜로 계산해 '짝지은 비교'를 만든다.
#
#  예산: 학습 33분 + 평가 5분 = 약 38분
# =====================================================================
import gc, time, math, numpy as np, torch
from torch.utils.data import DataLoader
from transformers import get_cosine_schedule_with_warmup
from tqdm.auto import tqdm

# ---------- 하이퍼파라미터 ----------
FT_SAMPLES   = 4000       # 시간 상한이 실질 제한. 넉넉히 잡고 시간으로 끊는다
FT_BATCH     = 1          # 7B 4bit + 640px + VRAM 여유 없음 -> 1
FT_ACCUM     = 8          # 유효 배치 8
FT_LR        = 1e-4
FT_MAX_MIN   = 33         # ★ 학습 시간 상한
EVAL_N       = 400        # 홀드아웃 앞 400건으로 검증 (표준오차 ±2.5%p)

import torch.nn.functional as F

# 노트북에서는 run_training() 안에 정의돼 있어 전역에 없으므로 여기서 다시 정의
def masked_weighted_loss(logits, labels, last, weight):
    """정답 위치 1개에 대한 CE를 샘플 가중치로 가중평균."""
    B = logits.size(0)
    r = torch.arange(B, device=logits.device)
    sel = logits[r, last - 1, :].float()      # 위치 t의 logits가 t+1을 예측
    tgt = labels[r, last]
    ce = F.cross_entropy(sel, tgt, reduction="none")
    return (ce * weight).sum() / weight.sum().clamp(min=1e-8)

CH = np.array(CHOICES)
gc.collect(); torch.cuda.empty_cache()
print(f"학습 전 VRAM 여유 {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

# ---------- 짝지은 비교 기준: 같은 400건의 zero-shot ----------
p_va_saved = np.load(ROOT / "vlm_probs_holdout_full_7B_4bit_640px_zs_nodev.npy").astype(np.float64)
va_eval  = va_part.head(EVAL_N).reset_index(drop=True)
gold_e   = va_eval["answer"].astype(str).str.strip().values
zs_e     = (CH[p_va_saved[:EVAL_N].argmax(1)] == gold_e).mean()
mc_e     = is_counting(va_eval["question"]).values
print(f"기준 zero-shot (같은 {EVAL_N}건) : {zs_e:.4f}"
      f"  (카운팅 {(CH[p_va_saved[:EVAL_N].argmax(1)][mc_e]==gold_e[mc_e]).mean():.4f})")

# ---------- 학습 ----------
ft_df = tr_part.head(FT_SAMPLES).reset_index(drop=True)
if "weight" not in ft_df.columns: ft_df["weight"] = 1.0
loader = DataLoader(VQADataset(ft_df, train=True, aug=USE_AUG), batch_size=FT_BATCH,
                    shuffle=True, collate_fn=Collator(processor, True),
                    num_workers=NUM_WORKERS, drop_last=False)
total = math.ceil(len(loader) / FT_ACCUM)
params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=FT_LR, weight_decay=0.01)
sch = get_cosine_schedule_with_warmup(opt, max(1, int(total * 0.03)), total)
print(f"학습 샘플 {len(ft_df)} / 최대 {total} step / 상한 {FT_MAX_MIN}분")

model.train(); opt.zero_grad(set_to_none=True)
t0, done, run, seen = time.time(), 0, 0.0, 0
pbar = tqdm(loader, desc="7B QLoRA", unit="b")
for step, batch in enumerate(pbar, 1):
    w    = batch.pop("_weight").to(device)
    last = batch.pop("_last").to(device)
    batch = {k: v.to(device) for k, v in batch.items() if isinstance(v, torch.Tensor)}
    lab = batch.pop("labels")
    loss = masked_weighted_loss(model(**batch).logits, lab, last, w)
    (loss / FT_ACCUM).backward()
    run += loss.item(); seen += 1
    if step % FT_ACCUM == 0 or step == len(loader):
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        opt.step(); sch.step(); opt.zero_grad(set_to_none=True); done += 1
        pbar.set_postfix({"loss": f"{run/max(seen,1):.3f}", "step": done})
        run, seen = 0.0, 0
        if (time.time() - t0) / 60 > FT_MAX_MIN:
            tqdm.write(f"\n⏰ {FT_MAX_MIN}분 경과 — 학습 종료 (step {done}, 샘플 {step})")
            break
print(f"\n학습 완료: {done} optimizer step / {time.time()-t0:.0f}초")

# ---------- 평가 ----------
gc.collect(); torch.cuda.empty_cache()
p_ft = predict_probs(va_eval, desc="학습후 평가")
ft_e = (CH[p_ft.argmax(1)] == gold_e).mean()

print("\n" + "=" * 54)
print(f"zero-shot   {zs_e:.4f}")
print(f"파인튜닝후   {ft_e:.4f}   ({(ft_e-zs_e)*100:+.2f}%p)")
print(f"  카운팅 {(CH[p_ft.argmax(1)][mc_e]==gold_e[mc_e]).mean():.4f}"
      f" / 기타 {(CH[p_ft.argmax(1)][~mc_e]==gold_e[~mc_e]).mean():.4f}")
print(f"표준오차 ±{100*(0.25/EVAL_N)**0.5:.1f}%p  (짝지은 비교라 실제 검출력은 더 높음)")
print("=" * 54)

np.save(ROOT / "vlm7b_probs_holdout_ft.npy", p_ft)
if ft_e > zs_e:
    model.save_pretrained(ROOT / "lora_7b_ft_check")
    print("어댑터 저장: lora_7b_ft_check  (집에서 이어서 학습 가능)")
else:
    print("zero-shot 미달 — 어댑터를 저장하지 않습니다.")
